# Phase 1: Data Collection & Crawling Pipeline

This notebook demonstrates the RSS feed and XML sitemap crawling pipeline used to collect the Vietnamese news dataset for clickbait detection. We pull headlines, snippets, and publication dates from 6 major Vietnamese news outlets:
- Tuổi Trẻ (`tuoitre`)
- Nhân Dân (`nhandan`)
- Thanh Niên (`thanhnien`)
- Kênh 14 (`kenh14`)
- Soha (`soha`)
- Afamily (`afamily`)

## Ethical Crawling & System Design
- **Polite Delays:** 1.0 second between consecutive requests to the same source, and 5.0 seconds between different sources.
- **Robots.txt Respect:** Checked before initiating crawling.
- **Resumable Progress:** A manifest checkpoint tracks crawl progress to prevent duplicate requests across runs.
- **Deterministic IDs:** MD5 hash generated from URLs to uniquely identify articles.

In [1]:
import sys
from pathlib import Path
import feedparser
import pandas as pd
import requests
import json
import xml.etree.ElementTree as ET

# Ensure project root is in system path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from configs.sources_config import RSS_SOURCES, SITEMAPS, SOURCE_CATEGORIES
print(f"Successfully imported sources config. Total sources configured: {len(RSS_SOURCES)}")

Successfully imported sources config. Total sources configured: 6


## 1. RSS Feed Exploration
Let's see how many feeds are configured for each source, and explore a sample of Vietnamese RSS content.

In [2]:
for source, feeds in RSS_SOURCES.items():
    print(f"Source: {source:<12} | Category: {SOURCE_CATEGORIES.get(source):<10} | Feeds: {len(feeds):>2}")

Source: tuoitre      | Category: formal_news | Feeds: 15
Source: nhandan      | Category: formal_news | Feeds: 15
Source: thanhnien    | Category: formal_news | Feeds: 15
Source: kenh14       | Category: entertainment | Feeds: 16
Source: soha         | Category: entertainment | Feeds: 13
Source: afamily      | Category: entertainment | Feeds: 10


Let's parse a sample feed from `tuoitre` using `feedparser` to inspect its structure.

In [3]:
sample_feed_url = RSS_SOURCES["tuoitre"][0]  # First feed for Tuổi Trẻ
print(f"Fetching sample feed: {sample_feed_url}")

headers = {"User-Agent": "Mozilla/5.0 (compatible; ClickbaitDatasetBot/1.0)"}
resp = requests.get(sample_feed_url, headers=headers, timeout=10)
feed = feedparser.parse(resp.content)

print(f"Feed Title: {feed.feed.get('title')}")
print(f"Number of entries: {len(feed.entries)}")

if len(feed.entries) > 0:
    entry = feed.entries[0]
    print("\nFirst Entry Metadata:")
    print(f"  - Title: {entry.get('title')}")
    print(f"  - Link:  {entry.get('link')}")
    print(f"  - Published Date: {entry.get('published')}")
    print(f"  - Description/Summary: {entry.get('summary')[:100]}...")

Fetching sample feed: https://tuoitre.vn/rss/tin-moi-nhat.rss


Feed Title: Tuổi Trẻ Online - Tin mới nhất - RSS Feed
Number of entries: 50

First Entry Metadata:
  - Title: Hoa hậu Ngọc Hân tặng áo dài, gây quỹ tặng 'Ước mơ của Thúy'
  - Link:  https://tuoitre.vn/hoa-hau-ngoc-han-tang-ao-dai-gay-quy-tang-uoc-mo-cua-thuy-20260614164619679.htm
  - Published Date: Sun, 14 Jun 2026 18:23:42 GMT+7
  - Description/Summary: <a href="https://tuoitre.vn/hoa-hau-ngoc-han-tang-ao-dai-gay-quy-tang-uoc-mo-cua-thuy-20260614164619...


## 2. Sitemap Crawling Demonstration
For sources without rich RSS history, we fall back to XML sitemaps to fetch historical URLs. Below is a demo of how we parse a sitemap index and extract URLs.

In [4]:
def extract_urls_from_sitemap(sitemap_url, max_urls=5):
    headers = {"User-Agent": "Mozilla/5.0 (compatible; ClickbaitDatasetBot/1.0)"}
    try:
        r = requests.get(sitemap_url, headers=headers, timeout=10)
        if r.status_code != 200:
            print(f"Failed to fetch sitemap: {sitemap_url} (HTTP {r.status_code})")
            return []
        
        root = ET.fromstring(r.content)
        # Handle XML namespaces if present
        ns = {"ns": "http://www.sitemaps.org/schemas/sitemap/0.9"}
        
        urls = []
        for loc in root.findall(".//ns:loc", ns):
            urls.append(loc.text)
            if len(urls) >= max_urls:
                break
        return urls
    except Exception as e:
        print(f"Error parsing sitemap: {e}")
        return []

if "nhandan" in SITEMAPS:
    sample_sitemap = SITEMAPS["nhandan"][0]
    print(f"Crawling sample sitemap from Nhân Dân: {sample_sitemap}")
    urls = extract_urls_from_sitemap(sample_sitemap)
    for idx, url in enumerate(urls):
        print(f"  [{idx+1}] {url}")

Crawling sample sitemap from Nhân Dân: https://nhandan.vn/sitemap.xml


  [1] https://nhandan.vn/sitemaps/categories.xml
  [2] https://nhandan.vn/sitemaps/topics.xml
  [3] https://nhandan.vn/sitemaps/news-2026-6.xml
  [4] https://nhandan.vn/sitemaps/google-news.xml
  [5] https://nhandan.vn/sitemaps/news-2026-5.xml


## 3. Crawled Dataset Checkpoint Manifest
A file-based manifest is saved under `.manifest/raw_crawl_manifest.json` to keep track of the crawl status and progress metrics.

In [5]:
manifest_path = project_root / ".manifest" / "raw_crawl_manifest.json"
if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    
    rss_feeds = 0
    rss_count = 0
    for src, feeds in manifest.get("rss", {}).items():
        for feed_url, info in feeds.items():
            rss_feeds += 1
            rss_count += info.get("count", 0)
            
    sitemap_urls = 0
    sitemap_count = 0
    for src, sitemaps in manifest.get("sitemap", {}).items():
        for sitemap_url, info in sitemaps.items():
            sitemap_urls += 1
            sitemap_count += info.get("count", 0)
            
    print(f"Manifest checkpoint loaded from {manifest_path.name}")
    print(f"RSS feeds tracked: {rss_feeds} feeds (Total fetched: {rss_count} articles)")
    print(f"Sitemaps tracked: {sitemap_urls} sitemaps (Total fetched: {sitemap_count} articles)")
else:
    print("No manifest found at .manifest/raw_crawl_manifest.json. Run python main.py --step 1 first.")

Manifest checkpoint loaded from raw_crawl_manifest.json
RSS feeds tracked: 84 feeds (Total fetched: 3870 articles)
Sitemaps tracked: 0 sitemaps (Total fetched: 0 articles)
